In [138]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import math

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dropout, LSTM, Dense, Layer, LayerNormalization, BatchNormalization, Conv1D, MaxPooling1D, Reshape, Add
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping

import yfinance as yf
import pandas_ta as ta

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option('display.max_columns', None)


In [112]:
def apply_ssa_trend(series, window = 15, lookback = 60):
    n_total = len(series)
    result = np.zeros(n_total)
    
    result[:lookback] = series[:lookback]
    
    for t in range(lookback, n_total):
        sub_series = series[t - lookback + 1 : t + 1]
        
        N = len(sub_series)
        L = window
        K = N - L + 1
        X = np.column_stack([sub_series[i:i+L] for i in range(K)])
        
        U, Sigma, VT = np.linalg.svd(X, full_matrices=False)
        
        X1 = Sigma[0] * np.outer(U[:, 0], VT[0, :])
        
        n = N - 1
        vals = [X1[i, n-i] for i in range(max(0, n-K+1), min(n+1, L))]
        result[t] = np.mean(vals)
        
    return result

def normalize_window(window):
    mean = window.mean(axis = 0)
    std = window.std(axis = 0) + 1e-8
    return (window - mean) / std

In [139]:
def get_dataset_columns(ticker, period="1y", window=60):
    df = yf.Tickers(ticker).history(period=period)
    vix_data = yf.Tickers("^VIX").history(period=period, progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    if isinstance(vix_data.columns, pd.MultiIndex):
        vix_data.columns = vix_data.columns.droplevel(1)

    df = df.ffill().dropna()
    vix_data = vix_data.reindex(df.index).ffill()

    open_trend = apply_ssa_trend(df['Open'].values, window=int((window/4)))
    vix_trend  = apply_ssa_trend(vix_data['Open'].values, window=int((window/4)))

    min_len = min(len(df), len(open_trend), len(vix_trend))
    df = df.iloc[:min_len]

    ########################################################################

    # Features
    df['Open_Trend'] = open_trend[:min_len]
    df['Vix_Trend'] = vix_trend[:min_len]
    df['Trend_Diff'] = df['Open'] - df['Open_Trend']
    df['Vix_Trend_Diff'] = vix_data['Open'] - df['Vix_Trend']

    df['RSI'] = ta.rsi(df['Close'], length=14)

    macd = ta.macd(df['Close'])
    df = pd.concat([df, macd], axis=1)

    bbands = ta.bbands(df['Close'], length=20)
    df = pd.concat([df, bbands], axis=1)

    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)

    df['Daily_Range'] = (df['High'] - df['Low']) / df['Open']

    df['VIX_Rel'] = vix_data['Close'] / vix_data['Close'].rolling(20).mean()

    ########################################################################

    # Target

    df['Target'] = (df['Open'].shift(-1) - df['Open']) / df['Open'] * 100
    # df['Target'] = np.log(df['Close'] / df['Open']) * 100
    # df['Target'] = df['Close']

    ########################################################################

    df = df.iloc[:-1]

    df = df.drop(df.index[0])
    df = df.drop(['High', 'Low', 'Volume', 'Close', 'Stock Splits', 'Dividends'], axis=1)
    
    df.columns.name = None
    exclude_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Target']
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    target_col = ["Target"]

    return df, feature_cols, target_col

In [114]:
def get_full_dataset(tickers, period="1y", window_size=60, test_size=0.2):
    X_total, y_total = [], []
    
    for ticker in tickers:
        df, feature_cols, target_col = get_dataset_columns(ticker, period=period, window=window_size)
        data_x = df[feature_cols].values
        data_y = df[target_col].values
        
        for i in range(len(df) - window_size):
            window_x = data_x[i : i + window_size]
            window_y = data_y[i + window_size]

            window_x = normalize_window(window_x)

            X_total.append(window_x)
            y_total.append(window_y)

    X_data = np.array(X_total)
    y_data = np.array(y_total)

    nan_mask = np.isnan(X_data).any(axis=(1, 2))
    X_data = X_data[~nan_mask]
    y_data = y_data[~nan_mask] 

    print(f"NaN Deleted: {np.sum(nan_mask)}")
    
    test_first_index = int(len(X_data) * (1 - test_size))
    X_train, X_test = X_data[:test_first_index], X_data[test_first_index:]
    y_train, y_test = y_data[:test_first_index], y_data[test_first_index:]

    print("""
    X_train Shape : {}
    X_test Shape : {}
    y_train Shape : {}
    y_test Shape : {}
    """.format(X_train.shape, X_test.shape, y_train.shape, y_test.shape))
    
    return X_train, X_test, y_train, y_test

In [115]:
def get_data_for_prediction(ticker, period="120d", window_size=60):
    df, feature_cols, _ = get_dataset_columns(ticker, period=period, window=window_size)

    if len(df) < window_size:
        raise ValueError("Not enough rows")

    X = df[feature_cols].iloc[-window_size:].values

    if np.isnan(X).any():
        raise ValueError("NaN detected in prediction window")

    X = normalize_window(X)

    X = X.reshape(1, window_size, len(feature_cols))

    return X

## Training

In [ ]:
tickers_5 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
tickers_20 = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", 
    "TSM", "AVGO", "ASML",
    "JPM", "BAC", "V", "MA",
    "JNJ", "LLY", "UNH", "PG", "KO", "PEP"
]
tickers_50 = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NFLX", "ADBE", "CRM", "ORCL",
    "NVDA", "TSM", "AVGO", "ASML", "AMD", "INTC", "QCOM", "TXN", "MU", "AMAT",
    "JPM", "BAC", "V", "MA", "GS", "MS", "AXP", "PYPL", "WFC", "BLK",
    "JNJ", "LLY", "UNH", "PFE", "ABBV", "MRK", "TMO", "DHR", "AMGN", "ISRG",
    "PG", "KO", "PEP", "WMT", "COST", "NKE", "MCD", "HD", "DIS", "SBUX"
]

# Get Dataset For Training
window_size = 60
X_train, X_test, y_train, y_test = get_full_dataset(
    tickers_50,
    period = "3y",
    window_size = window_size, 
    test_size = 0.2
)
feature_num = X_train.shape[-1]

In [ ]:
def get_CNN_LSTM_hybrid(window_size=60, feature_num=3):
    dropout_rate = 0.2
    
    input_layer = Input(shape=(window_size, feature_num), name="Input")
    
    x = Conv1D(filters=64, kernel_size=3, strides=1, padding='same', activation='relu', name="Conv1D_1")(input_layer)
    x = LayerNormalization(name="LayerNormalization_1")(x)
    x = Conv1D(filters=64, kernel_size=3, strides=1, padding='same', activation='relu', name="Conv1D_2")(x)
    x = LayerNormalization(name="LayerNormalization_2")(x)
    
    x = MaxPooling1D(pool_size=2, name="Max_Pooling")(x)
    
    x = LSTM(64, return_sequences=True, name="LSTM_1")(x)
    x = Dropout(dropout_rate, name="Dropout_1")(x)
    x = BatchNormalization(name="BatchNormalization_1")(x)
    
    x = LSTM(32, return_sequences=False, name="LSTM_2")(x)
    x = Dropout(dropout_rate, name="Dropout_2")(x)
    x = BatchNormalization(name="BatchNormalization_2")(x)
    
    x = Dense(16, activation="relu", name="FullyConnected_1")(x)
    output_layer = Dense(1, activation="linear", name="Output")(x)

    model = Model(inputs=input_layer, outputs=output_layer, name='SSA_CNN_LSTM_Hybrid')

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, 
                  loss='mse', 
                  metrics=['mae', tf.keras.metrics.RootMeanSquaredError()])
    return model

CNN_LSTM_model = get_CNN_LSTM_hybrid(window_size, feature_num)
CNN_LSTM_model.summary()

Model: "SSA_CNN_LSTM_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 60, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1D_1 (Conv1D)               │ (None, 60, 64)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LayerNormalization_1            │ (None, 60, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1D_2 (Conv1D)               │ (None, 60, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LayerNormalization_2            │ (None, 60, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Max_Pooling (MaxPooling1D)      │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LSTM_1 (LSTM)                   │ (None, 30, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BatchNormalization_1            │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ LSTM_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BatchNormalization_2            │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ FullyConnected_1 (Dense)        │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,617 (232.88 KB)

 Trainable params: 59,425 (232.13 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
earlyStopping_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

CNN_LSTM_model.fit(
    X_train, 
    y_train, 
    epochs = 50, 
    batch_size = 32,
    validation_data = (X_test, y_test),
    callbacks = [earlyStopping_callback]
)

Epoch 1/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - loss: 3.0637 - mae: 1.2302 - root_mean_squared_error: 1.7503 - val_loss: 1.4273 - val_mae: 0.8648 - val_root_mean_squared_error: 1.1947
Epoch 2/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - loss: 2.9999 - mae: 1.2102 - root_mean_squared_error: 1.7320 - val_loss: 1.4232 - val_mae: 0.8652 - val_root_mean_squared_error: 1.1930
Epoch 3/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - loss: 2.9807 - mae: 1.2036 - root_mean_squared_error: 1.7265 - val_loss: 1.4177 - val_mae: 0.8619 - val_root_mean_squared_error: 1.1907
Epoch 4/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - loss: 2.9714 - mae: 1.2013 - root_mean_squared_error: 1.7238 - val_loss: 1.4130 - val_mae: 0.8615 - val_root_mean_squared_error: 1.1887
Epoch 5/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - loss: 2.9707 - mae: 1.2019 - root_mean_squared_error: 1.7236 - val_loss: 1.4161 - val_mae: 0.8642 - val_root_mean_squared_error: 1.1900
Epoch 6/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
def get_backtest_data(ticker, period="2y", window_size=60):
    df = yf.Tickers(ticker).history(period=period, progress=False)
    vix_data = yf.Tickers("^VIX").history(period=period, progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    if isinstance(vix_data.columns, pd.MultiIndex):
        vix_data.columns = vix_data.columns.droplevel(1)
    
    df = df.ffill().dropna()
    vix_data = vix_data.reindex(df.index).ffill()

    open_trend = apply_ssa_trend(df['Open'].values, window=int((window_size/4)))
    vix_trend  = apply_ssa_trend(vix_data['Open'].values, window=int((window_size/4)))

    min_len = min(len(df), len(open_trend), len(vix_trend))

    df = df.iloc[:min_len]

    ############################################################

    # Columns

    df['Open_Trend'] = open_trend[:min_len]
    df['Vix_Trend'] = vix_trend[:min_len]
    df['Open_Trend_Diff'] = df['Open'] - df['Open_Trend']
    df['Vix_Trend_Diff'] = vix_data['Open'] - df['Vix_Trend']

    ############################################################

    # Targets

    df['Target'] = (df['Open'].shift(-1) - df['Open']) / df['Open'] * 100
    # df['Target'] = np.log(df['Close'] / df['Open']) * 100
    # df['Target'] = df['Close']

    ############################################################


    df = df.iloc[:-1]
    df = df.drop(df.index[0])
    df = df.drop(['High', 'Low', 'Volume', 'Stock Splits', 'Dividends'], axis=1)
    
    df.columns.name = None
    feature_cols = ["Open_Trend", "Trend_Diff", "Vix_Trend"]
    target_col = ["Target"]
    
    data_x = df[feature_cols].values
    data_y = df[target_col].values
    data_open_close = df[["Open", "Close"]].values

    X_list = []
    y_real = []
    dates = []
    actual_opens = []

    for i in range(len(df) - window_size):
        window_x = data_x[i : i + window_size]
        window_y = data_y[i + window_size]
        
        window_x = normalize_window(window_x)
        current_open = data_open_close[i + window_size]
        
        X_list.append(window_x)
        y_real.append(window_y)
        dates.append(df.index[i + window_size])
        actual_opens.append(current_open)

    return np.array(X_list), np.array(y_real), dates, feature_cols, np.array(actual_opens)

def backtest(model, ticker, period="1y", window_size=60, initial_capital=10000):
    X_bt, y_bt, bt_dates, feature_cols, data_open = get_backtest_data(ticker, period=period, window_size=window_size)
    predictions = model.predict(X_bt, verbose=0).flatten()
    
    cash = initial_capital
    shares = 0
    threshold = 0.05
    portfolio_history = []

    buy_count = 0
    buy_correct_count = 0
    actual_buy_count = 0
    sell_count = 0
    sell_correct_count = 0
    actual_sell_count = 0
    hold_count = 0
    hold_correct_count = 0
    actual_hold_count = 0

    correct_pred_count = 0
    wrong_pred_count = 0 
    
    for i in range(len(bt_dates)):
        current_open_price = data_open[i][0]
        y_act = y_bt[i][0]

        if y_act >= threshold:
            correct_signal = "BUY"
            actual_buy_count += 1
        elif y_act <= -1 * 0:
            correct_signal = "SELL"
            actual_sell_count += 1
        else:
            correct_signal ="HOLD"
            actual_hold_count += 1

        correct = False

        if predictions[i] >= threshold:
            signal = "BUY"
            buy_count += 1
            if correct_signal == "BUY":
                correct = True
        elif predictions[i] <= -1 * 0:
            signal = "SELL"
            sell_count += 1
            if correct_signal == "SELL":
                correct = True
        else:
            signal ="HOLD"
            hold_count += 1
            if correct_signal == "HOLD":
                correct = True
        
        if signal=="BUY" and cash > current_open_price:
            share_num = cash // current_open_price
            cash -= share_num * current_open_price
            shares += share_num

        elif signal=="SELL" and shares > 0:
            cash += shares * current_open_price
            shares = 0

        total_value = cash + (shares * current_open_price)

        if correct:
            correct_pred_count += 1

            if signal == "BUY":
                buy_correct_count += 1
            elif signal == "SELL":
                sell_correct_count += 1
            elif signal == "HOLD":
                hold_correct_count += 1
        else :
            wrong_pred_count += 1

        portfolio_history.append({
            'Date': bt_dates[i],
            'Signal': signal,
            'Co. Signal': correct_signal,
            'Correct': correct,

            'Open': current_open_price,
            'Actual': y_act,
            'Pred': predictions[i],
            'Shares': shares,
            'Cash': cash,
            'Total_Wallet': total_value,
            'Return (%)': ((total_value - initial_capital) / initial_capital) * 100,

            'SOLD when BUY':signal=="SELL" and correct_signal=="BUY",
            'BOUGHT when SELL': signal=="BUY" and correct_signal=="SELL",

            'Pred. Buys': buy_count,
            'Act. Buys': actual_buy_count,

            'Pred. Sells': sell_count,
            'Act. Sells': actual_sell_count,

            'Pred. Holds': hold_count,
            'Act. Holds': actual_hold_count,

            'Co. Buys': buy_correct_count,
            'Co. Sells': sell_correct_count,
            'Co. Holds': hold_correct_count,
            
            'Co. Dec': correct_pred_count,
            'Wr. Dec': wrong_pred_count,
        })

    print("Co.: Correct | Wr.: Wrong | Dec: Decision")
    result_df = pd.DataFrame(portfolio_history)
    return result_df

In [137]:
AAPL_Backtest = backtest(CNN_LSTM_model, "AAPL")
round(AAPL_Backtest, 4)

Co.: Correct | Wr.: Wrong | Dec: Decision


,Date,Signal,Co. Signal,Correct,Open,Actual,Pred,Shares,Cash,Total_Wallet,Return (%),SOLD when BUY,BOUGHT when SELL,Pred. Buys,Act. Buys,Pred. Sells,Act. Sells,Pred. Holds,Act. Holds,Co. Buys,Co. Sells,Co. Holds,Co. Dec,Wr. Dec
0,2025-06-03,BUY,BUY,True,200.7391,0.7748,0.0593,49.0,163.7825,10000.0000,0.0000,False,False,1,1,0,0,0,0,1,0,0,1,0
1,2025-06-04,BUY,BUY,True,202.2944,0.2908,0.0595,49.0,163.7825,10076.2084,0.7621,False,False,2,2,0,0,0,0,2,0,0,2,0
2,2025-06-05,BUY,SELL,False,202.8826,-0.2457,0.0509,49.0,163.7825,10105.0309,1.0503,False,True,3,2,0,1,0,0,2,0,0,2,1
3,2025-06-06,HOLD,BUY,False,202.3841,0.6847,0.0465,49.0,163.7825,10080.6046,0.8060,False,False,3,3,0,1,1,0,2,0,0,2,2
4,2025-06-09,HOLD,SELL,False,203.7699,-1.8543,0.0396,49.0,163.7825,10148.5086,1.4851,False,False,3,3,0,2,2,0,2,0,0,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,2026-02-26,BUY,SELL,False,274.9500,-0.7783,0.0589,46.0,17.9326,12665.6331,26.6563,False,True,75,101,30,79,80,5,42,14,2,58,127
185,2026-02-27,BUY,SELL,False,272.8100,-3.8122,0.0509,46.0,17.9326,12567.1925,25.6719,False,True,76,101,30,80,80,5,42,14,2,58,128
186,2026-03-02,BUY,BUY,True,262.4100,0.4078,0.0628,46.0,17.9326,12088.7928,20.8879,False,False,77,102,30,80,80,5,43,14,2,59,128
187,2026-03-03,BUY,BUY,True,263.4800,0.4440,0.0683,46.0,17.9326,12138.0131,21.3801,False,False,78,103,30,80,80,5,44,14,2,60,128


In [ ]:
def save_model_weights(model, file_name, folder_name):
    os.makedirs(folder_name, exist_ok=True)

    words = file_name.split(".")

    model_name = words[0]

    existing_files = [f for f in os.listdir(folder_name) if f.startswith(model_name)]
    next_number = len(existing_files) + 1
    words[0] = f"{model_name}_{next_number}"

    file_name = ".".join(words)
    save_path = os.path.join(folder_name, file_name)

    model.save_weights(save_path)
    print(f"model saved to: {save_path}")